# Chapter 4 — Training Models
Companion notebook to `part1_fundamentals/04_training_models.md`. Pure numpy/sklearn, no downloads needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import learning_curve
from sklearn import datasets

## The Normal Equation, by hand, vs scikit-learn

In [ ]:
np.random.seed(42)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)

X_b = np.c_[np.ones((100, 1)), X]
theta_best = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
print("Normal Equation theta:", theta_best.ravel())

lin_reg = LinearRegression()
lin_reg.fit(X, y)
print("sklearn theta:        ", [lin_reg.intercept_[0], lin_reg.coef_[0][0]])

## Gradient Descent (SGDRegressor)

In [ ]:
sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None, eta0=0.1, random_state=42)
sgd_reg.fit(X, y.ravel())
print("SGD theta:", [sgd_reg.intercept_[0], sgd_reg.coef_[0]])

## Polynomial Regression + Learning Curves (diagnose under/overfitting)

In [ ]:
np.random.seed(42)
m = 100
X_q = 6 * np.random.rand(m, 1) - 3
y_q = 0.5 * X_q ** 2 + X_q + 2 + np.random.randn(m, 1)

def plot_learning_curves(model, X, y, degree_label=""):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, scoring="neg_root_mean_squared_error",
        train_sizes=np.linspace(0.1, 1.0, 10))
    plt.plot(train_sizes, -train_scores.mean(axis=1), "r-+", label="train")
    plt.plot(train_sizes, -val_scores.mean(axis=1), "b-", label="validation")
    plt.legend(); plt.xlabel("Training set size"); plt.ylabel("RMSE")
    plt.title(f"Learning curves {degree_label}")
    plt.show()

for degree in [1, 2, 30]:
    poly_model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("lin_reg", LinearRegression()),
    ])
    plot_learning_curves(poly_model, X_q, y_q, degree_label=f"(degree={degree})")

## Regularization: Ridge, Lasso, Elastic Net

In [ ]:
for name, model in [("Ridge", Ridge(alpha=1)), ("Lasso", Lasso(alpha=0.1)),
                     ("ElasticNet", ElasticNet(alpha=0.1, l1_ratio=0.5))]:
    model.fit(X, y)
    print(f"{name:12s} coef={model.coef_}")

## Logistic & Softmax Regression on iris

In [ ]:
iris = datasets.load_iris()
X_iris = iris["data"][:, 3:]                  # petal width only
y_iris = (iris["target"] == 2).astype(int)     # 1 = Iris virginica

log_reg = LogisticRegression()
log_reg.fit(X_iris, y_iris)
print(log_reg.predict_proba([[1.7], [1.5]]))
print(log_reg.predict([[1.7], [1.5]]))

# Softmax (multinomial) on all 4 features, all 3 classes
X_all, y_all = iris["data"], iris["target"]
softmax_reg = LogisticRegression(multi_class="multinomial", solver="lbfgs", C=10)
softmax_reg.fit(X_all, y_all)
print("Softmax accuracy:", softmax_reg.score(X_all, y_all))

## ✍️ Practice
1. By hand: theta0=4, theta1=3, x1=2 -> predict y_hat.
2. Compare degree-1/2/300 polynomial fits + learning curves (done above) — describe what you see.
3. Why does Lasso zero out weights exactly, while Ridge only shrinks them close to zero?
4. Debug: SGD training loss oscillates wildly — name two likely causes + fixes.
5. Train softmax regression on all 4 iris features (done above) — inspect a few misclassified rows.

In [ ]:
# Your code / notes here